## Step1: Environmental preparation

In [7]:
import selenium
import os, re, sqlite3, time, random
import gzip
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import UnexpectedAlertPresentException

## Step2: config and database creating

3 tables
- `pages`: original page source codes - zip
- `stations`: stations' info
- `buses`: buses' info


In [ ]:
BASE_URL = "https://bis.dsat.gov.mo:37812/macauweb/"    # web url
ROUTES = ["26A", "3", "18"]     # routes of buses
INTERVAL_SECONDS = 600      # gap of each request (seconds), default = 600 (10min)
JITTER_SECONDS = 30     # random jitter (seconds)
DB_PATH = "macau_bus.db"        # databease - SQLite 
HEADLESS = False        # True = Headless mode; False = Show browser window

# string of current time
def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# create 3 tables in SQLite database
def create_tables(con):
    con.executescript('''
        CREATE TABLE IF NOT EXISTS pages (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            crawl_time TEXT NOT NULL,      -- time
            route_name TEXT NOT NULL,      -- route name
            direction  INTEGER NOT NULL,   -- direction - 0 or 1
            html       BLOB,               -- html source
            parsed     INTEGER DEFAULT 0   -- 0 = not parsed; 1 = parsed
        );
        CREATE TABLE IF NOT EXISTS stations (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            page_id       INTEGER NOT NULL,   -- which page
            seq           INTEGER,            -- order of station in the route, e.g., 0, 1, 2, ...
            station_code  TEXT,               -- station code, e.g., M1/13
            stopcode      TEXT,               -- station number, e.g., 00025001
            station_name  TEXT,               -- station name, e.g., 關閘總站
            lane_name     TEXT                -- lane name, e.g., C 車道
        );
        CREATE TABLE IF NOT EXISTS buses (
            id                   INTEGER PRIMARY KEY AUTOINCREMENT,
            page_id              INTEGER NOT NULL,   -- which page
            segment_seq          INTEGER,            -- vehicle position after which station
            segment_station_code TEXT,               -- station code after which vehicle is located
            bus_plate            TEXT,               -- license plate
            speed_kmh            REAL,               -- speed (km/h)
            position_pct         REAL                -- percentage position on this segment
        );
    ''')

# init database: open database. create tables if not exist, check compatibility
def init_db(path):
    con = sqlite3.connect(path)
    create_tables(con)
    # compatibility checking
    cols = [c[1] for c in con.execute("PRAGMA table_info(stations)")]
    if cols and "page_id" not in cols:
        con.close()
        os.replace(path, path + ".old")
        print("The new library is being rebuilt... %s" % (path + ".old"))
        con = sqlite3.connect(path)
        create_tables(con)
    return con

## Step3: crawl pages using Selenium, and save

In [12]:
# create browser driver. Selenium will automatically download the matching chromedriver.
def make_driver():
    opts = Options()
    opts.add_argument("--ignore-certificate-errors")  # ignore SSL certificate errors (this web must add)
    if HEADLESS:
        opts.add_argument("--headless=new")           # headless mode (no browser window)
    opts.add_argument("--disable-gpu")                # turn off GPU
    opts.add_argument("--window-size=1280,2000")      # window size
    opts.add_argument("--lang=zh-cn")                 # language
    opts.add_argument("--no-first-run")               # skip Chrome first setup
    d = webdriver.Chrome(options=opts)
    d.set_page_load_timeout(60)                       # give up if page load takes more than 60s
    return d

# construct the URL of a specific route and direction
# direction  --  0=outbound; 1=inbound.
def route_url(route, direction):
    return (BASE_URL + "routeLine.html?routeName=%s&direction=%d&language=zh-tw"
            "&ver=3.8.6&routeType=0&fromDzzp=false" % (route, direction))

# get the page source of a specific route and direction
# if the direction has no data - return None.
def get_page_source(driver, route, direction):
    driver.get(route_url(route, direction))       # open the page
    deadline = time.time() + 25                   # wait for 25 seconds at most
    while time.time() < deadline:
        try:
            # station list
            n = len(driver.find_elements(By.CSS_SELECTOR, "#bus_stations .macau-route-station"))
            if n > 0:
                time.sleep(4)                     # wait for 4s to let the page fully render
                return driver.page_source
        except UnexpectedAlertPresentException:
            driver.switch_to.alert.accept()
            return None
        time.sleep(1)                             # wait for 1s and try again
    return None                                   # wait for 25s and no data found

# save the page source into the database
def save_page(con, route, direction, html):
    gz = gzip.compress(html.encode("utf-8"))      # compress html source
    con.execute("INSERT INTO pages(crawl_time, route_name, direction, html, parsed) "
                "VALUES(?,?,?,?,0)", (now_str(), route, direction, gz))
    con.commit()

In [ ]:
# ---- test: 爬一条路线，把页面源码存进数据库 ----
con = init_db(DB_PATH)
driver = make_driver()
try:
    html = get_page_source(driver, ROUTES[0], 0)      # 爬 ROUTES[0] 的方向 0
    if html is None:
        print("网站暂时没返回数据，稍后重跑本格即可（不影响后面步骤）。")
    else:
        save_page(con, ROUTES[0], 0, html)
        print("已保存：%s 方向0，页面共 %d 个字符" % (ROUTES[0], len(html)))
        print("pages 表现在有 %d 条记录" %
              con.execute("SELECT COUNT(*) FROM pages").fetchone()[0])
except Exception as e:
    print("爬取失败：", e)
    print("（网站暂时不稳定，稍后重跑本格即可，不影响后面步骤）")
finally:
    driver.quit()

已保存：26A 方向0，页面共 206701 个字符
pages 表现在有 1 条记录


## Step4: regular expression

example
```JavaScript
<div id="_0" data-code="M1/13" data-stopcode="00025001" class="macau-route-station">
    ...
    <h4 ... station-code="M1/13" station-title="-關閘總站" station-lane-name="C 車道">
        <span id="M1/13" class="station-code 1">M1/13</span>-關閘總站（C 車道）</h4>
    ...
    <div class="busLocDom" style="top: 50%;">          ← 一辆车：位于"本站→下一站"的 50% 处
        <p class="busPlateL">AB4144</p>
        <div class="readyStart">27 km/h</div>
    </div>
</div>
<div id="_1" ...>...</div>   ← the next station
```

In [14]:
# extract stations and buses from the page source
def parse_route_page(html):
    stations = []
    buses = []

    # 1. match the start tag of each station
    # e.g., <div id="_0" data-code="M1/13" data-stopcode="00025001" class="macau-route-station">
    tag_re = re.compile(
        r'<div id="_(\d+)"'     # id="_0" - the number part
        r' data-code="([^"]+)"'     # data-code="M1/13" - station code
        r' data-stopcode="([^"]+)"'     # data-stopcode="00025001" - station number
        r' class="macau-route-station[^"]*">'       # class
    )
    tags = list(tag_re.finditer(html))

    # 2. processing each station
    for i, tag in enumerate(tags):
        seq  = int(tag.group(1))        # station sequence number
        code = tag.group(2)     # station code
        stop = tag.group(3)     # station number

        # content of this station = source code from the start tag of this site to the start tag of the next site
        end   = tags[i + 1].start() if i + 1 < len(tags) else len(html)
        block = html[tag.end():end]

        # 3. extract station and lane name
        # e.g., station-code="M1/13" station-title="-關閘總站" station-lane-name="C 車道"
        m = re.search(r'station-title="([^"]*)" station-lane-name="([^"]*)"', block)
        name = m.group(1).lstrip('-') if m else ''   # station name may have a "-" prefix, remove it
        lane = m.group(2) if m else ''
        stations.append({'seq': seq, 'code': code, 'stopcode': stop,
                         'name': name, 'lane': lane})

        # 4. extract buses at this station
        for bus_tag in re.finditer(r'<div class="busLocDom"', block):
            start = bus_tag.start()
            end2  = block.find('<div class="busLocDom"', start + 1)  # next bus start spot
            if end2 == -1:
                end2 = len(block)
            bus_block = block[start:end2]   # this bus's source code block

            # bus license plate: <p class="busPlateL" style="">AB4144</p>
            plate = re.search(r'<p class="busPlateL"[^>]*>([^<]+)</p>', bus_block)
            if not plate:
                continue
            # bus position: style="top: 50%;"  → 50, represents the bus is at 50% of this segment
            top   = re.search(r'style="top: ([0-9.]+)%', bus_block)
            # bus speed: <div class="readyStart">27 km/h</div>
            speed = re.search(r'<div class="readyStart">\s*([0-9.]+)\s*km/h', bus_block)

            buses.append({
                'page_seq': seq,        # this bus is between "this station" and "next station"
                'station_code': code,       # use "this station" to locate it
                'plate': plate.group(1).strip(),        # bus license plate
                'speed': float(speed.group(1)) if speed else None,      # speed
                'position': float(top.group(1)) if top else None,       # position percentage
            })

    return stations, buses

In [15]:
# parse pages that have not been extracted yet, and write the results into stations and buses tables
def parse_pages(con):
    rows = con.execute("SELECT id, html FROM pages WHERE parsed = 0").fetchall()
    for page_id, gz in rows:
        html = gzip.decompress(gz).decode("utf-8")          # unzip back to source code
        stations, buses = parse_route_page(html)            # regular extraction
        # wirte into stations table
        con.executemany(
            "INSERT INTO stations(page_id, seq, station_code, stopcode, station_name, lane_name) "
            "VALUES(?,?,?,?,?,?)",
            [(page_id, s['seq'], s['code'], s['stopcode'], s['name'], s['lane'])
             for s in stations])
        # write into buses table
        con.executemany(
            "INSERT INTO buses(page_id, segment_seq, segment_station_code, bus_plate, speed_kmh, position_pct) "
            "VALUES(?,?,?,?,?,?)",
            [(page_id, b['page_seq'], b['station_code'], b['plate'], b['speed'], b['position'])
             for b in buses])
        con.execute("UPDATE pages SET parsed = 1 WHERE id = ?", (page_id,))
        con.commit()
    return len(rows)

In [16]:
# ---- test: 把刚存进数据库的页面提取成结构化数据 ----
con = init_db(DB_PATH)
n = parse_pages(con)
print("本次提取了 %d 页" % n)

import pandas as pd
# 看看上一页（最近一条）提取出了什么
last = con.execute("SELECT id, route_name, direction FROM pages ORDER BY id DESC LIMIT 1").fetchone()
if last:
    print("\n最近一页：%s 方向%d\n" % (last[1], last[2]))
    sta = pd.read_sql_query(
        "SELECT seq, station_code, station_name FROM stations WHERE page_id=? ORDER BY seq", con, params=(last[0],))
    bus = pd.read_sql_query(
        "SELECT segment_station_code, bus_plate, speed_kmh, position_pct FROM buses WHERE page_id=?", con, params=(last[0],))
    print("站点（前 8 个）：")
    print(sta.head(8).to_string(index=False))
    print("\n车辆：")
    print(bus.to_string(index=False) if len(bus) else "（当前没有在线车辆，夜间属正常现象）")
con.close()

本次提取了 1 页

最近一页：26A 方向0

站点（前 8 个）：
 seq station_code station_name
   0        M95/3  沙梨頭北街/筷子基北灣
   1          M22     青洲/聖德蘭學校
   2          M10         台山街市
   3        M16/1     提督馬路/雅廉訪
   4         M109       提督/高士德
   5         M112   沙梨頭/華寶工業大廈
   6         M127         海邊新街
   7         M125        栢港停車場

车辆：
segment_station_code bus_plate  speed_kmh  position_pct
               M95/3    AB4650        NaN           NaN
                M127    AB4936       18.0           NaN
             M172/15    AB4819       47.0          50.0
                T375    AB4878        5.0          50.0
              T355/1    AB4950       18.0          50.0
